# 🏙️ SG Transit Liveability Index
### Real-time neighbourhood evaluation for Singapore flat hunters

**Goal:** Help you decide if a non-MRT district is worth moving into by analysing:
- 🚕 Real-time taxi availability + disappearance patterns
- 🚌 Bus frequency and reliability  
- 🏠 HDB resale prices by town
- 🗺️ Distance to nearest MRT station
- ⭐ Combined District Connectivity + Value-for-Money score

**Data sources:**
| Source | Data | Update frequency |
|--------|------|-----------------|
| LTA DataMall | Taxi positions, bus arrivals, bus stops | Every 60s / 3min |
| data.gov.sg | HDB resale prices (233k transactions) | Monthly |
| OneMap API | Geocoding, nearest MRT, basemap tiles | On demand |

**Pipeline architecture:**
```
LTA API → Ingestion → SQLite → Analytics → ML Forecasting → Dashboard
data.gov.sg → DuckDB → HDB Analytics → Map Page
OneMap API → Geocoding → Heatmap + Popup Cards
```

> Run each cell top to bottom. Mock mode works without API keys!


## ⚙️ Step 0 — One-time setup

### 0a — Install uv (run in PowerShell, NOT here)
```powershell
# Windows PowerShell
powershell -c "irm https://astral.sh/uv/install.sh | iex"
```
Then **restart your terminal**.

### 0b — Create and activate virtual environment
```powershell
cd "G:\My Drive\^SIT- SNAIC\Data Engineering project"
uv venv
.venv\Scripts\activate
uv pip install -r requirements.txt
```

### 0c — Set API keys permanently (do this once per machine)
1. Search **"Environment Variables"** in Windows Start
2. Click **"Edit the system environment variables"**
3. Click **"Environment Variables"** → **"New"** under User variables

| Variable name | Where to get it |
|--------------|----------------|
| `LTA_API_KEY` | [datamall.lta.gov.sg](https://datamall.lta.gov.sg) |
| `ONEMAP_TOKEN` | [developers.onemap.gov.sg](https://developers.onemap.gov.sg) |

### 0d — Seed historical data + train ML model (run once)
```powershell
python main.py --seed
```

### 0e — Geocode HDB addresses (run once, ~30 min with token)
```powershell
python hdb/geocoder.py
python hdb/quality_check.py --fix
```

### 0f — On a new machine
```powershell
cd "G:\My Drive\^SIT- SNAIC\Data Engineering project"
uv venv
.venv\Scripts\activate
uv pip install -r requirements.txt
```
Code + database sync via Google Drive. Just recreate the venv locally.

### 0g — Create .gitignore (keep secrets + big files out of GitHub)
```
.venv/
data/
__pycache__/
*.pyc
*.duckdb
*.db
```


In [ ]:
# Step 0h — Verify everything is ready
import sys, os

print(f"Python: {sys.version}")
in_venv = ".venv" in sys.executable or "Data Engineering" in sys.executable
print(f"Inside venv       : {'✅ Yes' if in_venv else '⚠️  No — activate venv first!'}")

# Check API keys
for key, url in [("LTA_API_KEY", "datamall.lta.gov.sg"), 
                  ("ONEMAP_TOKEN", "developers.onemap.gov.sg")]:
    val = os.environ.get(key, "")
    if val:
        print(f"{key:20s}: ✅ Set ({val[:6]}...{val[-4:]})")
    else:
        print(f"{key:20s}: ⚠️  Not set — get it from {url}")

# Check packages
packages = {
    "geopandas": "geospatial processing",
    "shapely":   "spatial buffers",
    "pandas":    "data manipulation", 
    "numpy":     "numerical computing",
    "sklearn":   "ML forecasting",
    "fastapi":   "REST API",
    "streamlit": "dashboard",
    "plotly":    "interactive charts",
    "apscheduler": "batch scheduling",
    "duckdb":    "HDB analytics",
    "requests":  "API calls",
}
print("\nPackage check:")
all_ok = True
for pkg, desc in packages.items():
    try:
        __import__(pkg)
        print(f"  ✅ {pkg:15s} ({desc})")
    except ImportError:
        print(f"  ❌ {pkg:15s} MISSING — run: uv pip install -r requirements.txt")
        all_ok = False

print("\n✅ All good — run the cells below!" if all_ok else "\n⚠️  Fix missing packages first!")


## 📦 Step 1 — Install Dependencies
Run this once. Takes ~2 min first time due to GeoPandas.

In [ ]:
import sys
!{sys.executable} -m pip install -q \
    geopandas shapely pyproj \
    pandas numpy scikit-learn \
    requests urllib3 \
    fastapi uvicorn pydantic \
    APScheduler \
    plotly ipywidgets

print("✅ All dependencies installed!")

## 🔧 Step 2 — Imports & Config

In [ ]:
import os
import time
import random
import sqlite3
import logging
import pickle
import threading
from pathlib import Path
from datetime import datetime, timedelta
from typing import Optional, Tuple
from collections import deque
from contextlib import contextmanager

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import plotly.express as px
import plotly.graph_objects as go

# Logging
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# ── District bounding boxes (MinLon, MaxLon, MinLat, MaxLat) ──────────────
BBOXES = {
    "marine_parade": (103.893, 103.935, 1.295, 1.316),   # non-MRT estate
    "downtown_cbd":  (103.845, 103.865, 1.277, 1.295),   # CBD reference
    "tengah":        (103.720, 103.760, 1.360, 1.390),   # new non-MRT town
}

# ── Your LTA API key (optional — mock mode works without it) ──────────────
LTA_API_KEY = os.environ.get("LTA_API_KEY", "")

print("✅ Config ready!")
print(f"   API key set: {'Yes ✓' if LTA_API_KEY else 'No — running in mock mode'}")
print(f"   Districts: {list(BBOXES.keys())}")

## 🗄️ Step 3 — SQLite Database
Creates `transport.db` in the current folder to persist all data.

In [ ]:
DB_PATH = Path("transport.db")

def init_db():
    conn = sqlite3.connect(str(DB_PATH))
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS taxi_snapshots (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            fetched_at  TEXT    NOT NULL,
            district    TEXT    NOT NULL,
            taxi_count  INTEGER NOT NULL,
            flux        REAL    DEFAULT 0,
            friction    REAL    DEFAULT 0
        );
        CREATE TABLE IF NOT EXISTS predictions (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            created_at      TEXT    NOT NULL,
            district        TEXT    NOT NULL,
            horizon_minutes INTEGER NOT NULL,
            predicted_count REAL    NOT NULL,
            actual_count    REAL,
            model_version   TEXT    DEFAULT 'v1'
        );
        CREATE TABLE IF NOT EXISTS anomaly_alerts (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            triggered_at TEXT    NOT NULL,
            district     TEXT    NOT NULL,
            alert_type   TEXT    NOT NULL,
            value        REAL    NOT NULL,
            threshold    REAL    NOT NULL,
            message      TEXT
        );
        CREATE TABLE IF NOT EXISTS model_metrics (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            evaluated_at TEXT    NOT NULL,
            district     TEXT    NOT NULL,
            mae          REAL,
            rmse         REAL,
            n_samples    INTEGER
        );
        CREATE INDEX IF NOT EXISTS idx_snap ON taxi_snapshots(district, fetched_at);
    """)
    conn.commit()
    conn.close()

def insert_snapshot(district, count, flux=0.0, friction=0.0):
    conn = sqlite3.connect(str(DB_PATH))
    conn.execute(
        "INSERT INTO taxi_snapshots (fetched_at, district, taxi_count, flux, friction) "
        "VALUES (?, ?, ?, ?, ?)",
        (datetime.utcnow().isoformat(), district, count, flux, friction)
    )
    conn.commit(); conn.close()

def fetch_snapshots(district, minutes=120):
    conn = sqlite3.connect(str(DB_PATH))
    rows = conn.execute(
        "SELECT fetched_at, taxi_count, flux, friction FROM taxi_snapshots "
        "WHERE district=? AND fetched_at >= datetime('now', ? || ' minutes') "
        "ORDER BY fetched_at ASC",
        (district, f"-{minutes}")
    ).fetchall()
    conn.close()
    return pd.DataFrame(rows, columns=["fetched_at","taxi_count","flux","friction"])

def fetch_alerts(district=None, limit=20):
    conn = sqlite3.connect(str(DB_PATH))
    if district:
        rows = conn.execute(
            "SELECT triggered_at, alert_type, message FROM anomaly_alerts "
            "WHERE district=? ORDER BY triggered_at DESC LIMIT ?",
            (district, limit)
        ).fetchall()
    else:
        rows = conn.execute(
            "SELECT district, triggered_at, alert_type, message FROM anomaly_alerts "
            "ORDER BY triggered_at DESC LIMIT ?", (limit,)
        ).fetchall()
    conn.close()
    return rows

init_db()
print(f"✅ Database ready at: {DB_PATH.absolute()}")

## 🌱 Step 4 — Seed Historical Data
Generates **7 days** of realistic synthetic taxi data so the ML model can train immediately.
Simulates morning/evening peaks, rain dips, and weekend patterns.
> If you have a real data.gov.sg CSV, upload it and we'll use that instead.


In [ ]:
def generate_synthetic_history(days=7):
    """Generate realistic synthetic taxi snapshots per district."""
    records = []
    now = datetime.utcnow()
    base_counts = {"marine_parade": 22, "downtown_cbd": 55, "tengah": 8}

    for district, base in base_counts.items():
        prev = base
        for mins_ago in range(days * 24 * 60, 0, -1):
            ts   = now - timedelta(minutes=mins_ago)
            hour = ts.hour

            # Time-of-day demand pattern
            if   7 <= hour < 10:  mult = 0.65   # morning peak (low availability)
            elif 17 <= hour < 20: mult = 0.60   # evening peak
            elif  0 <= hour <  5: mult = 1.35   # late night (high availability)
            elif 12 <= hour < 14: mult = 0.80   # lunch
            else:                 mult = 1.0

            if ts.weekday() >= 5: mult *= 1.15  # weekend uplift
            rain = -random.randint(5,12) if random.random() < 0.05 else 0
            count    = max(0, int(base * mult + random.gauss(0,2) + rain))
            flux     = float(count - prev)
            friction = round(min(1.0, abs(flux) / max(count,1) * 0.3), 4)
            records.append({
                "fetched_at": ts.isoformat(), "district": district,
                "taxi_count": count, "flux": round(flux,2), "friction": friction
            })
            prev = count

    return pd.DataFrame(records)

# Generate and bulk-insert
print("Generating 7 days of synthetic history...")
hist_df = generate_synthetic_history(days=7)

conn = sqlite3.connect(str(DB_PATH))
conn.execute("DELETE FROM taxi_snapshots")   # clear old data
rows = [(r.fetched_at, r.district, int(r.taxi_count), float(r.flux), float(r.friction))
        for r in hist_df.itertuples()]
conn.executemany(
    "INSERT INTO taxi_snapshots (fetched_at,district,taxi_count,flux,friction) "
    "VALUES (?,?,?,?,?)", rows
)
conn.commit(); conn.close()

print(f"✅ Inserted {len(rows):,} rows into database")
print(f"   Districts: {hist_df['district'].value_counts().to_dict()}")

## 🗺️ Step 5 — Geospatial Processing Engine

In [ ]:
CRS_WGS84 = "EPSG:4326"
CRS_SVY21 = "EPSG:3414"   # Singapore metre-based CRS for accurate buffers

BBox = Tuple[float, float, float, float]  # (MinLon, MaxLon, MinLat, MaxLat)

def parse_taxi_snapshot(payload: dict) -> gpd.GeoDataFrame:
    """Convert LTA GeoJSON payload to GeoDataFrame."""
    if not payload or "features" not in payload:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=CRS_WGS84)
    return gpd.GeoDataFrame.from_features(payload["features"], crs=CRS_WGS84)

def detect_disappearances(prev_gdf, curr_gdf, buffer_m=20.0):
    """
    Taxi Disappearance Engine.
    Re-projects to SVY21 (metres!) to apply an accurate 20m spatial buffer.
    Points from T-1 not covered by any buffer in T = Estimated Pickups.
    """
    if prev_gdf.empty or curr_gdf.empty:
        return gpd.GeoDataFrame(columns=["geometry","label"], crs=CRS_WGS84)
    prev_m    = prev_gdf.to_crs(CRS_SVY21)
    curr_m    = curr_gdf.to_crs(CRS_SVY21)
    covered   = curr_m.geometry.buffer(buffer_m).unary_union
    vanished  = prev_m[~prev_m.geometry.within(covered)].copy()
    vanished["label"] = "Estimated Pickup"
    return vanished.to_crs(CRS_WGS84)

def filter_taxis_in_bbox(gdf, bbox):
    min_lon, max_lon, min_lat, max_lat = bbox
    return gdf.cx[min_lon:max_lon, min_lat:max_lat].copy()

def filter_bus_stops_in_bbox(bus_df, bbox):
    min_lon, max_lon, min_lat, max_lat = bbox
    gdf = gpd.GeoDataFrame(
        bus_df,
        geometry=gpd.points_from_xy(bus_df["Longitude"], bus_df["Latitude"]),
        crs=CRS_WGS84,
    )
    return gdf.cx[min_lon:max_lon, min_lat:max_lat].copy()

def mock_taxi_gdf(n, bbox):
    """Generate n random taxi points inside bbox."""
    min_lon, max_lon, min_lat, max_lat = bbox
    features = [{
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [
            random.uniform(min_lon, max_lon),
            random.uniform(min_lat, max_lat),
        ]},
        "properties": {}
    } for _ in range(n)]
    return parse_taxi_snapshot({"type":"FeatureCollection","features":features})

def mock_bus_stops(bbox, n=10):
    min_lon, max_lon, min_lat, max_lat = bbox
    return pd.DataFrame([{
        "BusStopCode": f"9{i:04d}",
        "RoadName": f"Mock Road {i}",
        "Latitude":  random.uniform(min_lat, max_lat),
        "Longitude": random.uniform(min_lon, max_lon),
    } for i in range(n)])

def mock_bus_arrivals(bus_df):
    now, records = datetime.utcnow(), []
    for _, row in bus_df.iterrows():
        t1 = now + timedelta(minutes=random.uniform(1,5))
        t2 = t1  + timedelta(minutes=random.uniform(8,12))
        t3 = t2  + timedelta(minutes=random.uniform(8,12))
        records.append({
            "BusStopCode": row["BusStopCode"],
            "NextBus":  {"EstimatedArrival": t1.isoformat()},
            "NextBus2": {"EstimatedArrival": t2.isoformat()},
            "NextBus3": {"EstimatedArrival": t3.isoformat()},
        })
    return records

print("✅ Geospatial engine ready!")

# Quick test
test_gdf = mock_taxi_gdf(30, BBOXES["marine_parade"])
print(f"   Test: generated {len(test_gdf)} taxi points for Marine Parade")

## 📊 Step 6 — Analytics & Metrics Engine
Rolling 15-minute window tracking flux, friction, and bus reliability.

In [ ]:
WINDOW_MINUTES = 15
MAX_INTERVAL_S = 600.0
MAX_FLUX       = 20.0

class RollingWindow:
    def __init__(self, window_min=WINDOW_MINUTES):
        self._q = deque()
        self._w = timedelta(minutes=window_min)
    def push(self, data):
        self._q.append((datetime.utcnow(), data))
        self._evict()
    def items(self):
        self._evict()
        return list(self._q)
    def _evict(self):
        cutoff = datetime.utcnow() - self._w
        while self._q and self._q[0][0] < cutoff:
            self._q.popleft()
    def __len__(self): self._evict(); return len(self._q)

class MetricsEngine:
    def __init__(self):
        self._taxi = RollingWindow()
        self._bus  = RollingWindow()

    def ingest_taxi(self, gdf): self._taxi.push(gdf)
    def ingest_bus(self, arrivals): self._bus.push(arrivals)

    def taxi_flux(self, bbox):
        snaps = [g for _,g in self._taxi.items()]
        if len(snaps) < 2: return 0.0
        return float(
            len(filter_taxis_in_bbox(snaps[-1], bbox)) -
            len(filter_taxis_in_bbox(snaps[-2], bbox))
        )

    def taxi_friction(self, bbox):
        snaps = [g for _,g in self._taxi.items()]
        if len(snaps) < 2: return 0.0
        pickups = sum(
            len(detect_disappearances(
                filter_taxis_in_bbox(p, bbox),
                filter_taxis_in_bbox(c, bbox)
            ))
            for p, c in zip(snaps[:-1], snaps[1:])
        )
        current = len(filter_taxis_in_bbox(snaps[-1], bbox))
        return round(min(pickups / current, 1.0), 4) if current else 0.0

    def bus_reliability(self, bus_stops_gdf):
        if bus_stops_gdf.empty: return None
        stops  = set(bus_stops_gdf["BusStopCode"].astype(str))
        gaps   = []
        for _, batch in self._bus.items():
            for rec in batch:
                if str(rec.get("BusStopCode","")) not in stops: continue
                times = []
                for k in ("NextBus","NextBus2","NextBus3"):
                    eta = rec.get(k,{}).get("EstimatedArrival","")
                    if eta:
                        try: times.append(datetime.fromisoformat(eta))
                        except: pass
                times.sort()
                gaps += [(b-a).total_seconds() for a,b in zip(times[:-1],times[1:])]
        return round(float(np.mean(gaps)), 2) if gaps else None

    def latest_gdf(self):
        items = self._taxi.items()
        return items[-1][1] if items else None

    def snapshot_count(self): return len(self._taxi)

print("✅ Metrics engine ready!")

## 🤖 Step 7 — ML Taxi Forecaster
Ridge regression with rolling lag features. Predicts taxi count at +30, +60, +120 minutes.

In [ ]:
MODEL_DIR      = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
FORECAST_HORIZONS = [30, 60, 120]
FEATURE_COLS   = ["hour","minute","weekday",
                  "lag_1","lag_2","lag_3","lag_5","lag_10",
                  "roll5_mean","roll15_mean","roll5_std"]

def build_features(df):
    df = df.copy()
    df["fetched_at"] = pd.to_datetime(df["fetched_at"])
    df = df.set_index("fetched_at").sort_index()
    df = df[~df.index.duplicated(keep="last")]
    df["hour"]    = df.index.hour
    df["minute"]  = df.index.minute
    df["weekday"] = df.index.weekday
    for lag in [1,2,3,5,10]:
        df[f"lag_{lag}"] = df["taxi_count"].shift(lag)
    df["roll5_mean"]  = df["taxi_count"].rolling(5,  min_periods=1).mean()
    df["roll15_mean"] = df["taxi_count"].rolling(15, min_periods=1).mean()
    df["roll5_std"]   = df["taxi_count"].rolling(5,  min_periods=1).std().fillna(0)
    return df.dropna()

class TaxiForecaster:
    def __init__(self, district):
        self.district = district
        self._models  = {}

    def train(self, lookback_min=10080):  # 7 days default
        df_raw = fetch_snapshots(self.district, minutes=lookback_min)
        if len(df_raw) < 30:
            print(f"  [{self.district}] Only {len(df_raw)} rows — need ≥30 to train.")
            return {}
        df = build_features(df_raw)
        results = {}
        for h in FORECAST_HORIZONS:
            df["y"] = df["taxi_count"].shift(-h)
            train   = df.dropna(subset=["y"])
            if len(train) < 20: continue
            X, y = train[FEATURE_COLS].values, train["y"].values
            pipe = Pipeline([("sc", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
            pipe.fit(X, y)
            mae = mean_absolute_error(y, pipe.predict(X))
            self._models[h] = pipe
            results[h] = round(mae, 2)
            print(f"  [{self.district}] horizon={h}min  train_MAE={mae:.2f}  n={len(X)}")
        self._save()
        return results

    def predict(self):
        self._load()
        df_raw = fetch_snapshots(self.district, minutes=60)
        if df_raw.empty:
            return {h: 0.0 for h in FORECAST_HORIZONS}
        df  = build_features(df_raw)
        if df.empty:
            ema = float(df_raw["taxi_count"].ewm(span=10).mean().iloc[-1])
            return {h: round(ema,1) for h in FORECAST_HORIZONS}
        row = df[FEATURE_COLS].iloc[[-1]].values
        out = {}
        for h in FORECAST_HORIZONS:
            if h in self._models:
                out[h] = round(max(0.0, float(self._models[h].predict(row)[0])), 1)
            else:
                ema = float(df_raw["taxi_count"].ewm(span=10).mean().iloc[-1])
                out[h] = round(ema, 1)
            # Save to DB for later evaluation
            conn = sqlite3.connect(str(DB_PATH))
            conn.execute(
                "INSERT INTO predictions (created_at,district,horizon_minutes,predicted_count) "
                "VALUES (?,?,?,?)",
                (datetime.utcnow().isoformat(), self.district, h, out[h])
            )
            conn.commit(); conn.close()
        return out

    def evaluate(self):
        conn = sqlite3.connect(str(DB_PATH))
        rows = conn.execute(
            "SELECT predicted_count, actual_count FROM predictions "
            "WHERE district=? AND actual_count IS NOT NULL", (self.district,)
        ).fetchall()
        conn.close()
        if len(rows) < 5: return None
        df  = pd.DataFrame(rows, columns=["pred","actual"])
        mae  = mean_absolute_error(df.actual, df.pred)
        rmse = mean_squared_error(df.actual,  df.pred, squared=False)
        conn = sqlite3.connect(str(DB_PATH))
        conn.execute(
            "INSERT INTO model_metrics (evaluated_at,district,mae,rmse,n_samples) "
            "VALUES (?,?,?,?,?)",
            (datetime.utcnow().isoformat(), self.district, mae, rmse, len(df))
        )
        conn.commit(); conn.close()
        return {"mae": round(mae,3), "rmse": round(rmse,3), "n": len(df)}

    def _save(self):
        for h, m in self._models.items():
            with open(MODEL_DIR / f"{self.district}_{h}min.pkl","wb") as f: pickle.dump(m,f)

    def _load(self):
        if self._models: return
        for h in FORECAST_HORIZONS:
            p = MODEL_DIR / f"{self.district}_{h}min.pkl"
            if p.exists():
                with open(p,"rb") as f: self._models[h] = pickle.load(f)

print("✅ Forecaster class ready! Training now...")
for district in BBOXES:
    f = TaxiForecaster(district)
    f.train()

## 🚨 Step 8 — Anomaly Detection
Detects LOW_TAXI, HIGH_FLUX, and BUS_GAP events using rolling statistics.

In [ ]:
TAXI_SIGMA  = 2.0
FLUX_THRESH = 15
BUS_THRESH  = 480   # 8 minutes

def check_anomalies(district, current_count, flux, bus_interval=None):
    """Run all anomaly checks. Returns list of alert dicts."""
    alerts  = []
    history = fetch_snapshots(district, minutes=60)["taxi_count"].tolist()

    # LOW_TAXI: below rolling mean - 2σ
    if len(history) >= 10:
        mean, std = np.mean(history), np.std(history)
        lower = mean - TAXI_SIGMA * std
        if current_count < lower and current_count < mean * 0.5:
            msg = f"{district}: Only {current_count} taxis (mean={mean:.1f}, threshold={lower:.1f})"
            alerts.append({"type":"LOW_TAXI","value":current_count,"threshold":round(lower,2),"msg":msg})
            conn = sqlite3.connect(str(DB_PATH))
            conn.execute("INSERT INTO anomaly_alerts (triggered_at,district,alert_type,value,threshold,message) "
                         "VALUES (?,?,?,?,?,?)",
                         (datetime.utcnow().isoformat(),district,"LOW_TAXI",current_count,round(lower,2),msg))
            conn.commit(); conn.close()
            print(f"  🔴 {msg}")

    # HIGH_FLUX
    if abs(flux) >= FLUX_THRESH:
        direction = "surge" if flux > 0 else "drain"
        msg = f"{district}: Taxi {direction} of {flux:+.0f} (threshold=±{FLUX_THRESH})"
        alerts.append({"type":"HIGH_FLUX","value":flux,"threshold":FLUX_THRESH,"msg":msg})
        conn = sqlite3.connect(str(DB_PATH))
        conn.execute("INSERT INTO anomaly_alerts (triggered_at,district,alert_type,value,threshold,message) "
                     "VALUES (?,?,?,?,?,?)",
                     (datetime.utcnow().isoformat(),district,"HIGH_FLUX",flux,FLUX_THRESH,msg))
        conn.commit(); conn.close()
        print(f"  🟡 {msg}")

    # BUS_GAP
    if bus_interval and bus_interval > BUS_THRESH:
        msg = f"{district}: Bus gap {bus_interval:.0f}s exceeds {BUS_THRESH}s"
        alerts.append({"type":"BUS_GAP","value":bus_interval,"threshold":BUS_THRESH,"msg":msg})
        conn = sqlite3.connect(str(DB_PATH))
        conn.execute("INSERT INTO anomaly_alerts (triggered_at,district,alert_type,value,threshold,message) "
                     "VALUES (?,?,?,?,?,?)",
                     (datetime.utcnow().isoformat(),district,"BUS_GAP",bus_interval,BUS_THRESH,msg))
        conn.commit(); conn.close()
        print(f"  🔵 {msg}")

    if not alerts:
        print(f"  ✅ {district}: No anomalies (count={current_count}, flux={flux:+.0f})")
    return alerts

print("✅ Anomaly detector ready!")

## 🏙️ Step 9 — District Connectivity Score
Weighted formula: `(Bus Frequency × 0.5) + (Taxi Stability × 0.3) − (Friction × 0.2)`

In [ ]:
def evaluate_district(bbox, engine, bus_stops_df):
    """
    Returns a 0-100 District Connectivity Score.
    Formula: (bus_freq*0.5) + (taxi_stability*0.3) - (friction*0.2)
    """
    flux        = engine.taxi_flux(bbox)
    friction    = engine.taxi_friction(bbox)
    bus_stops   = filter_bus_stops_in_bbox(bus_stops_df, bbox)
    bus_interval = engine.bus_reliability(bus_stops)

    bus_freq_score     = max(0., 1. - (bus_interval or MAX_INTERVAL_S) / MAX_INTERVAL_S) * 100
    taxi_stability     = max(0., 1. - abs(flux) / MAX_FLUX) * 100
    friction_penalty   = friction * 100

    score = (bus_freq_score * 0.5) + (taxi_stability * 0.3) - (friction_penalty * 0.2)
    score = round(max(0., min(100., score)), 2)

    return {
        "score":             score,
        "taxi_flux":         flux,
        "taxi_friction":     friction,
        "bus_interval_s":    bus_interval,
        "bus_freq_score":    round(bus_freq_score, 2),
        "taxi_stability":    round(taxi_stability, 2),
        "friction_penalty":  round(friction_penalty, 2),
    }

def rank_districts(engine, bboxes):
    """
    Tier 1 bonus: score all districts and return sorted leaderboard.
    """
    results = []
    for name, bbox in bboxes.items():
        bus_df  = mock_bus_stops(bbox)
        result  = evaluate_district(bbox, engine, bus_df)
        results.append({"district": name, **result})
    results.sort(key=lambda x: x["score"], reverse=True)
    for i, r in enumerate(results, 1):
        r["rank"] = i
    return results

print("✅ Evaluator ready!")

## 🚀 Step 10 — Run the Full Pipeline
Feeds mock snapshots through every layer and shows the district leaderboard.

In [ ]:
print("=" * 60)
print("  RUNNING FULL PIPELINE — MOCK MODE")
print("=" * 60)

engine = MetricsEngine()

# Feed two snapshots per district into engine
for district, bbox in BBOXES.items():
    count1 = random.randint(20, 40)
    snap1  = mock_taxi_gdf(count1, bbox)
    engine.ingest_taxi(snap1)
    insert_snapshot(district, count1)

    count2 = random.randint(15, 35)
    snap2  = mock_taxi_gdf(count2, bbox)
    engine.ingest_taxi(snap2)

    bus_df   = mock_bus_stops(bbox)
    arrivals = mock_bus_arrivals(bus_df)
    engine.ingest_bus(arrivals)

    flux = float(count2 - count1)
    insert_snapshot(district, count2, flux=flux)

    print(f"\n[{district}]")
    check_anomalies(district, count2, flux)

# Rank all districts
print("\n" + "─"*50)
print("📊 DISTRICT LEADERBOARD")
print("─"*50)
ranked = rank_districts(engine, BBOXES)
medals = ["🥇","🥈","🥉"]
for r in ranked:
    print(f"  {medals[r['rank']-1]} #{r['rank']} {r['district']:20s}  "
          f"Score: {r['score']:5.1f}  "
          f"Flux: {r['taxi_flux']:+.0f}  "
          f"Friction: {r['taxi_friction']:.3f}")

# ML Predictions
print("\n" + "─"*50)
print("🔮 ML PREDICTIONS")
print("─"*50)
for district in BBOXES:
    f    = TaxiForecaster(district)
    pred = f.predict()
    print(f"  [{district}] " + "  ".join(f"+{h}min→{v:.1f}" for h,v in pred.items()))

print("\n✅ Pipeline complete!")

## 📈 Step 11 — Visualise the Data

In [ ]:
# ── Chart 1: Taxi history per district ────────────────────────────────────
fig = go.Figure()
colors = {"marine_parade":"#1E88E5", "downtown_cbd":"#43A047", "tengah":"#FB8C00"}

for district, color in colors.items():
    df = fetch_snapshots(district, minutes=180)
    if df.empty: continue
    df["fetched_at"] = pd.to_datetime(df["fetched_at"])
    df["roll_mean"]  = df["taxi_count"].rolling(10, min_periods=1).mean()
    df["roll_std"]   = df["taxi_count"].rolling(10, min_periods=1).std().fillna(0)

    fig.add_trace(go.Scatter(
        x=df["fetched_at"], y=df["taxi_count"],
        mode="lines", name=district.replace("_"," ").title(),
        line=dict(color=color, width=2), opacity=0.7
    ))
    fig.add_trace(go.Scatter(
        x=pd.concat([df["fetched_at"], df["fetched_at"].iloc[::-1]]),
        y=pd.concat([df["roll_mean"]+2*df["roll_std"],
                     (df["roll_mean"]-2*df["roll_std"]).iloc[::-1]]),
        fill="toself", fillcolor=color.replace(")",",0.08)").replace("rgb","rgba"),
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False, name=f"{district} ±2σ"
    ))

fig.update_layout(
    title="Taxi Availability History by District",
    xaxis_title="Time (UTC)", yaxis_title="Taxi Count",
    height=400, template="plotly_white",
    legend=dict(orientation="h", y=-0.2)
)
fig.show()

# ── Chart 2: District score leaderboard ────────────────────────────────────
rank_df = pd.DataFrame(ranked)
fig2 = px.bar(
    rank_df, x="district", y="score",
    color="score", color_continuous_scale=["#E53935","#FFA726","#43A047"],
    range_y=[0,100], text="score",
    title="District Connectivity Score Leaderboard",
    labels={"district":"District","score":"Score (0-100)"}
)
fig2.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig2.update_layout(height=350, template="plotly_white", showlegend=False)
fig2.show()

# ── Chart 3: Flux over time ────────────────────────────────────────────────
fig3 = go.Figure()
for district, color in colors.items():
    df = fetch_snapshots(district, minutes=180)
    if df.empty or "flux" not in df.columns: continue
    df["fetched_at"] = pd.to_datetime(df["fetched_at"])
    fig3.add_trace(go.Bar(
        x=df["fetched_at"].tail(40), y=df["flux"].tail(40),
        name=district.replace("_"," ").title(), marker_color=color, opacity=0.7
    ))
fig3.update_layout(
    title="Taxi Flux (Inflow/Outflow) by District",
    xaxis_title="Time", yaxis_title="Flux",
    barmode="group", height=350, template="plotly_white"
)
fig3.show()

## 🔑 Step 12 — Live LTA API (Optional)
Only run this if you have an LTA API key. Skip if running in mock mode.


In [ ]:
import requests as req
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def build_lta_session(api_key):
    s = req.Session()
    s.headers.update({"AccountKey": api_key, "accept": "application/json"})
    retry = Retry(total=4, backoff_factor=2, status_forcelist=[429,500,502,503,504])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s

def fetch_live_taxis(api_key):
    s    = build_lta_session(api_key)
    resp = s.get("https://datamall2.mytransport.sg/ltaodataservice/Taxi-Availability", timeout=10)
    resp.raise_for_status()
    return resp.json()

if LTA_API_KEY:
    print("Fetching live taxi data...")
    payload  = fetch_live_taxis(LTA_API_KEY)
    live_gdf = parse_taxi_snapshot(payload)
    print(f"✅ Live taxis in Singapore right now: {len(live_gdf)}")
    for district, bbox in BBOXES.items():
        in_area = filter_taxis_in_bbox(live_gdf, bbox)
        print(f"   {district}: {len(in_area)} taxis")
else:
    print("⚠️  No LTA_API_KEY set. To use live data:")
    print("    import os")
    print("    os.environ['LTA_API_KEY'] = 'your_key_here'")
    print("    LTA_API_KEY = os.environ['LTA_API_KEY']")
    print("    Then re-run this cell.")

## 🚀 What's coming next — Airflow + Docker

### Current scheduler (APScheduler)
Right now we use APScheduler built into Python:
```
Every 08:00 SGT → retrain ML model
Every 08:05 SGT → evaluate predictions  
Every 30 min    → predictions + anomaly check
```

### Upgrading to Apache Airflow
Airflow is the industry standard for data pipeline orchestration. It gives us:
- Visual DAG (Directed Acyclic Graph) UI
- Task dependency management
- Retry on failure with alerts
- Full run history and logs

Our pipeline as an Airflow DAG:
```
ingest_taxi → process_disappearances → update_metrics → run_predictions → check_anomalies
                                                      ↘ daily_retrain (08:00 SGT)
```

### Docker setup (coming soon)
Docker packages the whole app into containers so it runs anywhere:
```yaml
# docker-compose.yml (planned)
services:
  pipeline:    # main.py — ingestion + API
  dashboard:   # streamlit dashboard
  airflow:     # scheduler + DAG UI
  db:          # SQLite + DuckDB volumes
```

One command to start everything:
```bash
docker-compose up
```

### Timeline
| Feature | Status |
|---------|--------|
| Transport pipeline | ✅ Done |
| HDB integration | ✅ Done |
| Singapore map | 🔜 Next |
| Nearest MRT API | 🔜 Next |
| Block popup cards | 🔜 Next |
| Docker | 📋 Planned |
| Airflow | 📋 Planned |
| dbt transforms | 📋 Planned |


## ✅ What we built — SG Transit Liveability Index

| Layer | Component | What it does |
|-------|-----------|-------------|
| **Ingestion** | TaxiWorker | LTA taxi positions every 60s |
| **Ingestion** | BusWorker | LTA bus arrivals every 3 min |
| **Storage** | SQLite transport.db | Taxi snapshots, alerts, predictions, ML metrics |
| **Storage** | DuckDB hdb.duckdb | 233k HDB resale transactions + geocache |
| **Processing** | Taxi disappearance engine | SVY21 reprojection + 20m spatial buffer |
| **Processing** | OneMap geocoder | 9,712 HDB blocks → lat/lng coordinates |
| **Analytics** | Metrics engine | Flux, friction, bus headway, connectivity score |
| **Analytics** | HDB analytics | Town price summaries, Value-for-Money score |
| **ML** | Ridge forecaster | +30/60/120 min taxi predictions |
| **ML** | Anomaly detector | LOW_TAXI, HIGH_FLUX, BUS_GAP alerts |
| **ML** | Batch scheduler | Daily retrain at 08:00 SGT |
| **API** | FastAPI | /evaluate, /rank, /predictions, /alerts |
| **Dashboard** | Streamlit | Live charts, forecasts, bus section, leaderboard |
| **Dashboard** | Glossary page | Plain English explanations |
| **Dashboard** | Singapore map | HDB heatmap + MRT overlay (🔜) |
| **Dashboard** | Block popup card | All data in one card (🔜) |

### Run the full pipeline
```bash
# Terminal 1 — start pipeline + API
python main.py

# Terminal 2 — open dashboard  
streamlit run dashboard/app.py

# Browser
open http://localhost:8501
```

### GitHub
[github.com/minna711/sg-transit-liveability](https://github.com/minna711/sg-transit-liveability)
